每个通道都有一个卷积核，结果是所有通道卷积结果的和

无论有多少输入通道，到目前为止我们只用到单输出通道。

我们可以有多个三位卷积核，每个核生成一个输出通道

这样每个输出通道可以识别特定模式，输入通道核识别并组合输入中的模式。

# 多输入输出通道

## 1. 为什么需要通道
灰度图 1 通道；彩色 RGB 3 通道；上层卷积输出也是多通道。
输入带通道后，单个 2-D 核不够用。

## 2. 多输入通道：核变厚
- 每个输入通道配一个自己的 2-D 核，各做各的互相关
- 结果逐元素相加 → 输出仍是单通道
- 核从 2-D 升级为 3-D：(输入通道数, 核高, 核宽)

## 3. 多输出通道：一套核 → 一张图
- 相加后只剩 1 张 → 想要几张输出就配几套核
- 每套核（含每通道一个）输出一张特征图
- 每个输出通道专门识别一种模式

## 4. 完整规则
几个输入通道 → 每套核有几个成员
几个输出通道 → 总共几套核

## 5. 4-D 权重落地
weight.shape = (out_channels, in_channels, kH, kW)
- 前两维是"通道"（几套 × 每套几个），后两维是核的空间尺寸
- 参数量 = out × in × kH × kW
- 通道是参数量的主要来源

## 6. 1×1 卷积
- 核 1×1，不看空间邻域
- 只在每个像素位置对所有通道加权求和
- 等价于"通道上的全连接"，用于混合/升降维

## 7. 核心结论
- 输入通道的核负责识别各通道的局部模式，
  输出通道的整套核负责组合成更高级特征。
- 之前 4-D 权重 (out,in,kH,kW) 的谜底：前两维 = 通道，后两维 = 空间。

In [2]:
import torch
from d2l import torch as d2l

def corr2d_multi_in(X,K):
    return sum(d2l.corr2d(x,k) for x,k in zip(X,K))

验证互相关运算的输出

In [3]:
X =torch.tensor([[[0.0,1.0,2.0],[3.0,4.0,5.0],[6.0,7.0,8.0]],
                 [[1.0,2.0,3.0],[4.0,5.0,6.0],[7.0,8.0,9.0]]])
K=torch.tensor([[[0.0,1.0],[2.0,3.0]],[[1.0,2.0],[3.0,4.0]]])
corr2d_multi_in(X,K)

tensor([[ 56.,  72.],
        [104., 120.]])

计算多个通道的输出的互相关函数

In [4]:
def corr2d_multi_in_out(X,K):
    return torch.stack([corr2d_multi_in(X,k) for k in K],0)
K=torch.stack((K,K+1,K+2),0)
K.shape

torch.Size([3, 2, 2, 2])

In [5]:
corr2d_multi_in_out(X,K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

1x1卷积

In [6]:
def corr2d_multi_in_out_lxl(X,K):
    c_i,h,w=X.shape
    c_o=K.shape[0]
    X=X.reshape((c_i,h*w))
    K=K.reshape((c_o,c_i))
    Y=torch.matmul(K,X)
    return Y.reshape((c_o,h,w))
X=torch.normal(0,1,(3,3,3))
K=torch.normal(0,1,(2,3,1,1))
Y1=corr2d_multi_in_out_lxl(X,K)
Y2=corr2d_multi_in_out(X,K)
float(torch.abs(Y1-Y2).sum())<1e-6

True